# Stock Trend Prediction - CNN & LSTM Pipeline

This notebook covers:
1. **Phase 1: Data Preparation & Chart Generation** - Fetching historical data from Yahoo Finance (last **10 years → today**), calculating daily returns, relative volume, and generating 20-day chart images.
2. **Phase 2: Model Development & Training** - Splitting data by time (**Train:** 10 yrs ago → 2 yrs ago | **Test:** 2 yrs ago → today), labeling data based on day-21 returns, and training both a CNN on images and an LSTM on raw sequences.
3. **Phase 3: Inference & Evaluation** - Evaluating accuracy, saving models (`.h5`), rewriting test images with predictions, and providing a function to randomly display test results.

In [1]:
import subprocess
venv_pip = r'C:\Users\ELCOT\.gemini\antigravity\scratch\options-iq\venv312\Scripts\pip.exe'
subprocess.check_call([venv_pip, 'install', '--quiet', 'yfinance', 'tensorflow', 'Pillow', 'matplotlib', 'pandas', 'numpy', 'scikit-learn'])
print('All packages installed successfully!')

All packages installed successfully!


In [2]:
import os
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from PIL import Image
from datetime import datetime
import shutil
import random

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, LSTM, BatchNormalization
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

## Configuration & Data Fetching

In [3]:
from datetime import datetime, timedelta

# ── Dynamic date configuration (always relative to today) ──────────────
today      = datetime.today()
END_DATE   = today.strftime('%Y-%m-%d')                          # today
START_DATE = today.replace(year=today.year - 10).strftime('%Y-%m-%d')  # 10 yrs back
_train_end = today.replace(year=today.year - 2)                  # 2 yrs back
TRAIN_END  = _train_end.strftime('%Y-%m-%d')
TEST_START = (_train_end + timedelta(days=1)).strftime('%Y-%m-%d')

print(f'Date Range : {START_DATE}  -->  {END_DATE}')
print(f'Train      : {START_DATE}  -->  {TRAIN_END}')
print(f'Test       : {TEST_START}  -->  {END_DATE}')

# ── Tickers ────────────────────────────────────────────────────────────
TICKERS = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'TSLA', 'NVDA',
    'JPM',  'BAC',  'GS',   'JNJ',  'PFE',  'XOM',  'CVX',
    'WMT',  'HD',   'DIS',  'NFLX', 'AMD',  'INTC'
]

WINDOW    = 20
THRESHOLD = 0.005  # +/- 0.5 % return for labels

# ── Output directories ─────────────────────────────────────────────────
import os
IMAGE_DIR = 'chart_images'
TRAIN_DIR = os.path.join(IMAGE_DIR, 'train')
TEST_DIR  = os.path.join(IMAGE_DIR, 'test')
PRED_DIR  = os.path.join(IMAGE_DIR, 'predictions')

for d in [TRAIN_DIR, TEST_DIR, PRED_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Fetch data ─────────────────────────────────────────────────────────
import yfinance as yf
import pandas as pd
all_data = {}
print('Fetching data...')
for ticker in TICKERS:
    df = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False)
    if not df.empty:
        if hasattr(df.columns, 'levels'):
            df.columns = df.columns.get_level_values(0)
        df['Daily_Return'] = df['Close'].pct_change().fillna(0)
        all_data[ticker] = df

print(f'Fetched data for {len(all_data)} tickers.')


Date Range : 2016-05-05  -->  2026-05-05
Train      : 2016-05-05  -->  2024-05-05
Test       : 2024-05-06  -->  2026-05-05
Fetching data...
Fetched data for 20 tickers.


## Phase 1: Data Preparation & Chart Generation
We generate 20-day charts for CNN and raw sequences for LSTM. Labels are: -1 (Fall), 0 (Neutral), 1 (Rise).

In [4]:


def get_label(ret, threshold):
    if ret > threshold: return 1
    elif ret < -threshold: return -1
    else: return 0

def generate_chart(cum_returns, rel_volume, filename):
    matplotlib.use("Agg")
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(2.24, 2.24), dpi=100, gridspec_kw={"height_ratios": [2, 1]})
    
    # Top panel: cumulative returns
    ax1.plot(cum_returns, color="black", linewidth=1)
    ax1.set_xticks([])
    ax1.set_yticks([])
    
    # Bottom panel: relative volume
    ax2.bar(range(len(rel_volume)), rel_volume, color="gray", width=0.8)
    ax2.set_xticks([])
    ax2.set_yticks([])
    
    for ax in [ax1, ax2]:
        for spine in ax.spines.values():
            spine.set_visible(False)
            
    plt.subplots_adjust(hspace=0.05, left=0, right=1, top=1, bottom=0)
    fig.savefig(filename, format="png", bbox_inches="tight", pad_inches=0)
    plt.close(fig)

cnn_data = {'train': [], 'test': []}
lstm_data = {'train': [], 'test': []}

print("Generating data... This will take a few minutes.")
for ticker, df in all_data.items():
    prices = df['Close'].values
    volumes = df['Volume'].values
    dates = df.index
    daily_rets = df['Daily_Return'].values
    
    # We only take every 5th day to avoid massive overlap and speed up training
    for i in range(WINDOW, len(df)-1, 5):
        window_dates = dates[i-WINDOW:i]
        target_date = dates[i]
        target_ret = daily_rets[i]
        
        end_date_str = window_dates[-1].strftime('%Y%m%d')
        
        # Features for window
        w_rets = daily_rets[i-WINDOW:i]
        w_vols = volumes[i-WINDOW:i]
        
        cum_rets = np.cumprod(1 + w_rets) - 1
        avg_vol = np.mean(w_vols)
        rel_vol = w_vols / avg_vol if avg_vol > 0 else np.ones_like(w_vols)
        
        label = get_label(target_ret, THRESHOLD)
        split = 'train' if window_dates[-1] <= pd.to_datetime(TRAIN_END) else 'test'
        
        # Prepare LSTM Sequence
        seq = np.stack([w_rets, rel_vol], axis=-1)
        lstm_data[split].append({'seq': seq, 'label': label})
        
        # Generate Image for CNN
        folder = TRAIN_DIR if split == 'train' else TEST_DIR
        filename = os.path.join(folder, f"{ticker}_{end_date_str}_{label}.png")
        generate_chart(cum_rets, rel_vol, filename)
        cnn_data[split].append({'file': filename, 'label': label, 'ticker': ticker, 'date': end_date_str})
        
print("Data generation complete.")
print(f"CNN Train Images: {len(cnn_data['train'])}, Test Images: {len(cnn_data['test'])}")
print(f"LSTM Train Seq: {len(lstm_data['train'])}, Test Seq: {len(lstm_data['test'])}")

Generating data... This will take a few minutes.
Data generation complete.
CNN Train Images: 7980, Test Images: 2000
LSTM Train Seq: 7980, Test Seq: 2000


## Phase 2: Model Development & Training

### Loading Data for Training

In [5]:
def load_cnn_data(data_list):
    X, y = [], []
    for item in data_list:
        img = Image.open(item['file']).convert('RGB').resize((64, 64))
        X.append(np.array(img, dtype=np.float32) / 255.0)
        # Map label -1,0,1 to 0,1,2
        y.append(item['label'] + 1) 
    return np.array(X), to_categorical(np.array(y), num_classes=3)

print("Loading CNN data to memory...")
X_cnn_train, y_cnn_train = load_cnn_data(cnn_data['train'])
X_cnn_test, y_cnn_test = load_cnn_data(cnn_data['test'])

def load_lstm_data(data_list):
    X, y = [], []
    for item in data_list:
        X.append(item['seq'])
        y.append(item['label'] + 1)
    return np.array(X), to_categorical(np.array(y), num_classes=3)

X_lstm_train, y_lstm_train = load_lstm_data(lstm_data['train'])
X_lstm_test, y_lstm_test = load_lstm_data(lstm_data['test'])

Loading CNN data to memory...


### Train CNN Model

In [6]:
def build_cnn():
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=(64, 64, 3)),
        MaxPooling2D(2,2),
        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D(2,2),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(3, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

cnn_model = build_cnn()
print("Training CNN...")
cnn_history = cnn_model.fit(
    X_cnn_train, y_cnn_train,
    validation_data=(X_cnn_test, y_cnn_test),
    epochs=10,
    batch_size=32,
    verbose=1
)

cnn_loss, cnn_acc = cnn_model.evaluate(X_cnn_test, y_cnn_test, verbose=0)
print(f"\nCNN Test Accuracy: {cnn_acc*100:.2f}%")
cnn_model.save('stock_cnn_model.h5')
print("CNN model saved.")

Training CNN...
Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 24s 82ms/step - accuracy: 0.3813 - loss: 1.1060 - val_accuracy: 0.4050 - val_loss: 1.0883
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 40s 77ms/step - accuracy: 0.3812 - loss: 1.0937 - val_accuracy: 0.4050 - val_loss: 1.0875
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 19s 76ms/step - accuracy: 0.3812 - loss: 1.0933 - val_accuracy: 0.4050 - val_loss: 1.0870
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 19s 77ms/step - accuracy: 0.3812 - loss: 1.0937 - val_accuracy: 0.4050 - val_loss: 1.0883
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 22s 81ms/step - accuracy: 0.3812 - loss: 1.0937 - val_accuracy: 0.4050 - val_loss: 1.0887
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 19s 76ms/step - accuracy: 0.3812 - loss: 1.0937 - val_accuracy: 0.4050 - val_loss: 1.0880
Epoch 7/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 20s 79ms/step - accuracy: 0.3812 - loss: 1.0935 - val_accuracy: 0.4050 - val_loss: 1.0872
Epoch 8/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 21s 82ms/step - accuracy: 0.3812 -


CNN Test Accuracy: 40.50%
CNN model saved.


### Train LSTM Model

In [7]:
def build_lstm():
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(WINDOW, 2)),
        Dropout(0.3),
        LSTM(32),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(3, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

lstm_model = build_lstm()
print("Training LSTM...")
lstm_history = lstm_model.fit(
    X_lstm_train, y_lstm_train,
    validation_data=(X_lstm_test, y_lstm_test),
    epochs=15,
    batch_size=32,
    verbose=1
)

lstm_loss, lstm_acc = lstm_model.evaluate(X_lstm_test, y_lstm_test, verbose=0)
print(f"\nLSTM Test Accuracy: {lstm_acc*100:.2f}%")
lstm_model.save('stock_lstm_model.h5')
print("LSTM model saved.")

Training LSTM...
Epoch 1/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 12s 22ms/step - accuracy: 0.3778 - loss: 1.0957 - val_accuracy: 0.4050 - val_loss: 1.0868
Epoch 2/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.3801 - loss: 1.0949 - val_accuracy: 0.4050 - val_loss: 1.0874
Epoch 3/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.3812 - loss: 1.0943 - val_accuracy: 0.4050 - val_loss: 1.0879
Epoch 4/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.3812 - loss: 1.0940 - val_accuracy: 0.4050 - val_loss: 1.0889
Epoch 5/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.3812 - loss: 1.0938 - val_accuracy: 0.4050 - val_loss: 1.0868
Epoch 6/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.3812 - loss: 1.0938 - val_accuracy: 0.4050 - val_loss: 1.0869
Epoch 7/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.3812 - loss: 1.0939 - val_accuracy: 0.4050 - val_loss: 1.0883
Epoch 8/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.3811 - loss:


LSTM Test Accuracy: 40.50%
LSTM model saved.


## Phase 3: Inference & Evaluation
Run inference on test images, rename them with predictions, and create display function.

In [8]:
print("Running inference on test images to generate predictions...")
cnn_preds = cnn_model.predict(X_cnn_test, verbose=0)
cnn_pred_classes = np.argmax(cnn_preds, axis=1) - 1 # back to -1,0,1

for i, item in enumerate(cnn_data['test']):
    orig_file = item['file']
    ticker = item['ticker']
    date_str = item['date']
    true_label = item['label']
    pred_label = cnn_pred_classes[i]
    
    new_filename = f"{ticker}_{date_str}_{true_label}_{pred_label}.png"
    new_filepath = os.path.join(PRED_DIR, new_filename)
    shutil.copy(orig_file, new_filepath)
    item['pred_file'] = new_filepath
    item['pred_label'] = pred_label

print(f"Saved {len(cnn_data['test'])} prediction images to {PRED_DIR}.")

Running inference on test images to generate predictions...
Saved 2000 prediction images to chart_images\predictions.


In [9]:
def show_image(stock, label, prediction):
    """
    Finds and displays a random image from the test set matching criteria.
    """
    candidates = [x for x in cnn_data['test'] 
                  if x['ticker'] == stock and x['label'] == label and x['pred_label'] == prediction]
    
    if not candidates:
        print(f"No images found for {stock} with Label={label} and Prediction={prediction}.")
        return
    
    selected = random.choice(candidates)
    img = Image.open(selected['pred_file'])
    
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"{stock} | Date: {selected['date']} | Label: {label} | Pred: {prediction}")
    plt.show()

# Example usage:
show_image('MSFT', 0, 1)
show_image('AAPL', 1, 1)
show_image('TSLA', -1, -1)

No images found for TSLA with Label=-1 and Prediction=-1.


Finally, copy models to the backend models folder so they can be used.

In [10]:
os.makedirs('backend/models', exist_ok=True)
shutil.copy('stock_cnn_model.h5', 'backend/models/')
shutil.copy('stock_lstm_model.h5', 'backend/models/')
print("Copied models to backend/models/")

Copied models to backend/models/
